### Goals
- This module focuses on multi-agent workflows
  

#### Parallelization 


In [ ]:
import os
import sys
from operator import add
from typing import Annotated, Any, TypedDict

from IPython.display import Image, display
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph

sys.path.append(os.path.abspath(os.path.join("..")))
from dotenv import load_dotenv
from langchain_community.document_loaders import WikipediaLoader
from tinyfish import TinyFish

from utils.utils import create_tavily

load_dotenv()

True

In [ ]:
os.environ["TINYFISH_API_KEY"] = os.getenv("TINYFISH_API_KEY") or ""

In [ ]:
tiny_client = TinyFish()
response = tiny_client.search.query(query="what is langgraph")

In [ ]:
for r in response.results:
    print(f"{r.title} -> {r.url}")

In [ ]:
loader = WikipediaLoader(query="Large language models", load_max_docs=5)
docs = loader.load()
docs

In [ ]:
tavily_client = create_tavily()
response = tavily_client.search(
    query="how is nvidia results for Q1 26, only the main points", max_results=3
)

In [ ]:
response["results"]

In [ ]:
for result in response["results"]:
    print(f"{result['title']}\n{result['content'][:500]}")

In [ ]:
loader = WikipediaLoader(query="langchain", lang="en", load_max_docs=1)
docs = loader.load()
print(docs[0].page_content[:500])

In [ ]:
class State(TypedDict):
    state: str


class NodeReturnValue:
    def __init__(self, node_secret: str) -> None:
        self.value = node_secret

    def __call__(self, state: State) -> Any:
        print(f"Adding {self.value} to {state['state']}")
        state["state"] = self.value
        return {"state": [self.value]}

In [ ]:
builder = StateGraph(State)
builder.add_node("a", NodeReturnValue("I'm A"))
builder.add_node("b", NodeReturnValue("I'm B"))
builder.add_node("c", NodeReturnValue("I'm C"))
builder.add_node("d", NodeReturnValue("I'm D"))
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", "c")
builder.add_edge("c", "d")
builder.add_edge("d", END)
graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"state": []})

##### **Fan out and fan in**

In [ ]:
builder_1 = StateGraph(State)
builder_1.add_node("a", NodeReturnValue("I am A"))
builder_1.add_node("b", NodeReturnValue("I am B"))
builder_1.add_node("c", NodeReturnValue("I am C"))
builder_1.add_node("d", NodeReturnValue("I am D"))
builder_1.add_edge(START, "a")
builder_1.add_edge("a", "b")
builder_1.add_edge("a", "c")
builder_1.add_edge("c", "d")
builder_1.add_edge("b", "d")
builder_1.add_edge("d", END)
graph_1 = builder_1.compile()
display(Image(graph_1.get_graph().draw_mermaid_png()))

In [ ]:
try:
    graph_1.invoke({"state": []})
except InvalidUpdateError as e:
    print(f"Error {e}")

- In the above graph both node `b` and `c` try to update node `d` at the same time which raises an error
- For this we  have to use annotated and a reducer function, in the state

In [ ]:
class UpdatedState(TypedDict):
    state: Annotated[list, add]


In [ ]:
builder_2 = StateGraph(UpdatedState)
builder_2.add_node("a", NodeReturnValue("I am A"))
builder_2.add_node("b", NodeReturnValue("I am B"))
builder_2.add_node("c", NodeReturnValue("I am C"))
builder_2.add_node("d", NodeReturnValue("I am D"))
builder_2.add_edge(START, "a")
builder_2.add_edge("a", "b")
builder_2.add_edge("a", "c")
builder_2.add_edge("c", "d")
builder_2.add_edge("b", "d")
builder_2.add_edge("d", END)
graph_2 = builder_2.compile()
display(Image(graph_2.get_graph().draw_mermaid_png()))

In [ ]:
graph_2.invoke({"state": []})

In [ ]:
builder_3 = StateGraph(UpdatedState)
builder_3.add_node("a", NodeReturnValue("I am A"))
builder_3.add_node("b1", NodeReturnValue("I am B1"))
builder_3.add_node("b2", NodeReturnValue("I am B2"))
builder_3.add_node("c", NodeReturnValue("I am C"))
builder_3.add_node("d", NodeReturnValue("I am D"))
builder_3.add_edge(START, "a")
builder_3.add_edge("a", "b1")
builder_3.add_edge("b1", "b2")
builder_3.add_edge("b2", "d")
builder_3.add_edge("a", "c")
builder_3.add_edge(["b2", "c"], "d")
builder_3.add_edge("d", END)
graph_3 = builder_3.compile()
display(Image(graph_3.get_graph().draw_mermaid_png()))


- The graph execution waits for the all the nodes in a branch to finish before going to the next node
- Here the graph waits for all the nodes on the left to finish execution before proceeding to `D` but that's because we explicitly made it clear that node `b2` and `c` are on the same level with this line `builder_3.add_edge(["b2", "c"], "d")`

In [ ]:
graph_3.invoke({"state": []})

#### __*Setting the order of State Updates*__
- Within each step we do not have specific control over the state updates
- In simple terms the exection is *deterministic order* that __*you do not control*__
- However, we can use custom reducer to customize the order i.e, sort state updates

In [ ]:
def sort_reducer(left, right):
    """Cobines and sorts the values in a list"""
    if not isinstance(left, list):
        left = [left]
    if not isinstance(right, list):
        right = [right]
    return sorted(left + right, reverse=False)

In [ ]:
class UpdatedReducer(TypedDict):
    state: Annotated[list, sort_reducer]

In [ ]:
builder_4 = StateGraph(UpdatedReducer)
builder_4.add_node("a", NodeReturnValue("I am A"))
builder_4.add_node("b1", NodeReturnValue("I am B1"))
builder_4.add_node("b2", NodeReturnValue("I am B2"))
builder_4.add_node("c", NodeReturnValue("I am C"))
builder_4.add_node("d", NodeReturnValue("I am D"))
builder_4.add_edge(START, "a")
builder_4.add_edge("a", "b1")
builder_4.add_edge("b1", "b2")
builder_4.add_edge("b2", "d")
builder_4.add_edge("a", "c")
builder_4.add_edge(["b2", "c"], "d")
builder_4.add_edge("d", END)
graph_4 = builder_4.compile()
display(Image(graph_3.get_graph().draw_mermaid_png()))


In [ ]:
graph_4.invoke({"state": []})

#### __Working with LLMs__

In [ ]:
# TODO Create the graph to parallel execute two nodes tavily and tinyfish